# Indoor passages and platforms

The interiors of many stations are mapped in OpenStreetMap. The walkways between the entrances and the platforms carry a `level` tag, and stairs, escalators and lifts join the levels. With `indoor_elements=True`, `get_network` and `get_graph` add these to a walking network, so that a route can run from the street down to a platform. Here, we read the walking network around Victoria and Green Park stations in London and draw it in 3D with `pyrosm.plot_3d`. The drawing needs plotly, which you can install with `pip install "pyrosm[3d]"`.

## Read the network of a station

The extract of Greater London comes from Geofabrik (about 130 MB). A bounding box around the two stations keeps the read small.

In [1]:
from pyrosm import OSM, get_data

fp = get_data("greater_london")
osm = OSM(fp, bounding_box=[-0.150, 51.492, -0.136, 51.510])
nodes, edges = osm.get_network("walking", nodes=True, indoor_elements=True)

`indoor_elements=True` needs `nodes=True` and adds to the walking network:

- the platforms (`railway=platform`, `public_transport=platform` and `highway=platform`), even when the network type would otherwise leave them out;
- lines across platform areas and indoor areas (`indoor=corridor` or `indoor=area`), each on its own level. These areas are crossed in the same way as the squares on the page [Walking areas](walking_areas.ipynb), and the lines across them have `True` in the column `walking_area`;
- the columns `u_level` and `v_level`, which hold the level at each end of an edge, from the OSM `level` tags. An edge tagged with one level has that level at both ends. Stairs and escalators run between the levels of the walkways they join. For a walkway tagged with several levels, such as `level=0;1`, pyrosm cannot tell which level each end is on, so both columns are NaN. They are NaN for an edge without a `level` tag too;
- the column `vertical`: `escalator`, `stairs` or `lift`;
- lifts split per level. A lift node where edges on several levels meet becomes one node per level, and lift edges of length 0 join these nodes. The node on the lowest level keeps the lift node's id, and the others get negative ids derived from it.

OpenStreetMap stores no heights, so the `length` of an edge comes from its coordinates on the map. For escalators, pyrosm estimates the length along the slope instead. The escalator standards EN 115-1 and ASME A17.1 allow an angle of at most 30°. By default, pyrosm divides the map length of an escalator by cos 30°, so its length is about 1.155 times its length on the map. `escalator_slope` sets another angle, and `escalator_slope=None` keeps the map length. Stairs keep their length on the map, and lift edges have length 0.

In [2]:
edges["vertical"].value_counts()

vertical
stairs       91
escalator    24
lift         10
Name: count, dtype: int64

In [3]:
columns = ["id", "osm_type", "u", "v", "vertical", "u_level", "v_level", "length"]
edges.loc[edges["vertical"] == "lift", columns].head()

,id,osm_type,u,v,vertical,u_level,v_level,length
7503,12279370522,node,12279370522,-12279370522001,lift,-4.0,-3.0,0.0
7504,12279370533,node,12279370533,-12279370533001,lift,-4.0,-3.0,0.0
7505,12279370579,node,12279370579,-12279370579001,lift,-3.0,-2.0,0.0
7506,12279370579,node,-12279370579001,-12279370579002,lift,-2.0,-1.0,0.0
7507,13618518471,node,13618518471,-13618518471001,lift,-3.0,-1.0,0.0


## Draw a station in 3D

`plot_3d` draws the edges at their levels. Walkways, stairs, escalators, lifts, platforms and the lines across areas each have an entry in the legend. Click an entry to hide or show those edges. The vertical axis counts levels, not metres. With `center` and `radius`, only the edges that reach within that many metres of the center are drawn. The station entrances among the nodes are drawn as markers.

In [4]:
from pyrosm import plot_3d

victoria = (-0.14315, 51.49637)
fig = plot_3d(edges, nodes, center=victoria, radius=200)
fig.show()

In a script, `fig.show(renderer="browser")` opens the view in a browser window, and `fig.write_html("victoria.html")` saves it as a web page.

## Follow a route from the street to a platform

`get_graph(indoor_elements=True)` keeps stairs, escalators and lifts as separate edges, also in a simplified graph. Here, we use networkx to find the shortest walk from a street entrance down to the southbound platform of the Victoria line, at level -4. The walk starts at the entrance node 7730374800 and ends at node 5442230281 on the platform. `plot_3d` draws the route as a band.

In [5]:
import networkx as nx

G = osm.get_graph("walking", graph_type="networkx", indoor_elements=True)
route = nx.shortest_path(G, 7730374800, 5442230281, weight="length")
fig = plot_3d(G, center=victoria, radius=200, route=route)
fig.show()

To time the walk, we divide the `length` of each edge by a speed. The speeds in this example are assumptions. We assume a walking speed of 4.5 km/h (1.25 m/s). We also use this speed for stairs; we did not find a source for a typical speed on stairs. For the escalator, we assume 0.65 m/s. The escalator maker KONE recommends this speed for railway and metro stations where the escalators climb a large height ({cite:label}`kone2020`). The escalators of a particular station may run at another speed. Both the escalator speed and the edge's `length` are measured along the steps.

In [6]:
import pandas as pd

walking_speed = 4.5 / 3.6  # m/s
escalator_speed = 0.65  # m/s

steps = []
for u, v in zip(route[:-1], route[1:]):
    edge = min(G[u][v].values(), key=lambda data: data["length"])
    escalator = edge.get("vertical") == "escalator"
    speed = escalator_speed if escalator else walking_speed
    steps.append(
        {
            "vertical": edge.get("vertical"),
            "u_level": edge.get("u_level"),
            "v_level": edge.get("v_level"),
            "length": edge["length"],
            "seconds": round(edge["length"] / speed, 1),
        }
    )
steps = pd.DataFrame(steps)
steps

,vertical,u_level,v_level,length,seconds
0,stairs,0.0,-1.0,13.114,10.5
1,NaN,-1.0,-1.0,29.058,23.2
2,escalator,-1.0,-4.0,23.257,35.8
3,NaN,-4.0,-4.0,12.206,9.8


In [7]:
print("%.0f m in %.0f s" % (steps["length"].sum(), steps["seconds"].sum()))

78 m in 79 s


## Green Park

The same read holds Green Park station, about 1.1 km north of Victoria.

In [8]:
green_park = (-0.14294, 51.50662)
fig = plot_3d(edges, nodes, center=green_park, radius=170)
fig.show()

## Write the network to a PBF file

`OSM.write_pbf` keeps the lifts split per level when it writes these frames to a PBF file. Each copy of a lift node becomes a new node, and each lift edge becomes a new way tagged `highway=elevator`. A way that meets a lift ends at the copy on its own level. `write_pbf` reports these new elements in a `pyrosm.Notice`.